# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: CTR / Engagement Opportunity Scoring**

I'm choosing this lane because the starter data shows a clean, position-adjusted signal I can defend with numbers, not intuition. CTR isn't comparable across pages unless you first control for average position — a page at position 1 should get far more clicks than a page at position 9 for the exact same content quality. That structure (expected CTR varies by tier, then compare pages *within* their own tier) is exactly the kind of pattern that's "real but too messy to write by hand" across thousands of pages: a human reviewer can't mentally hold 30,000 tier-adjusted comparisons, but a ranked, reason-coded queue can surface the worst gaps first.

(First check in the data cell below already caught a real gotcha: `position_tier` lumps `avg_position == 0` — the dataset's "no data" sentinel — into the `top_3` bucket. Filtering those out before computing tier-level CTR is step one of this lane, not an afterthought.)

## 2. The question: decision, action, cost of a wrong call

**Unit of analysis:** one content item / page (`content_id`), described by its trailing-90-day metrics.

**The decision:** Which already-visible pages (they're ranking and getting impressions) are under-capturing clicks relative to what their position tier normally earns — and should an SEO/content reviewer look at first?

**Who acts, and what they do:** A content/SEO reviewer on the FlyRank team (or client-side) pulls the top of the ranked queue and reviews title, meta description, and snippet structure for those pages first, since those are the levers that move CTR without touching rankings.

**Cost of a wrong call:** If I flag a page that isn't really underperforming (e.g. its low CTR is just noise from a handful of impressions, or its intent doesn't match a clickable SERP feature), the reviewer spends limited time rewriting a page that gains nothing — and that's time not spent on a page that actually needed it. The cost isn't catastrophic (nobody loses money directly), but reviewer hours are the scarce resource this whole project is meant to protect, so a noisy queue quietly wastes the exact thing I'm trying to save.

**Why data/ML helps:** A flat CTR threshold ("flag anything under 2%") would systematically punish pages that rank at position 8 and reward pages that rank at position 2, because expected CTR falls off sharply and non-linearly with position. Comparing pages only to *their own tier's* expected CTR is more than an if-statement can cleanly express once you also want to rank by gap size and back it with volume filters — that's a scoring/ranking problem, not a single rule.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Total rows:", len(df))

# Gotcha check first: avg_position == 0 means "no data", NOT rank zero (per data dictionary).
# The position_tier column lumps these into "top_3" by default, which would silently wreck
# any tier-based CTR comparison -- exactly the kind of mistake this lane depends on avoiding.
no_position_data = (df["avg_position"] == 0).sum()
print(f"Rows with avg_position == 0 (no position data, must exclude): {no_position_data}")

has_position = df[df["avg_position"] > 0]

# 1. How much does expected CTR really vary by position tier, once "no data" rows are removed?
# (This is the whole premise of the lane -- if it didn't vary, tier-adjustment would be pointless.)
tier_ctr = has_position.groupby("position_tier")["ctr"].agg(["count", "mean", "median"]).sort_values("median", ascending=False)
print("\nCTR by position tier (avg_position > 0 only):")
print(tier_ctr)

# 2. How many pages are even eligible: visible (ranking, position 1-20) with enough
# impression volume (>=500) to make a CTR comparison meaningful, not noise?
visible_enough_volume = has_position[(has_position["avg_position"] <= 20) & (has_position["impressions_90d"] >= 500)]
print("\nVisible pages with enough volume (position 1-20, impressions_90d >= 500):", len(visible_enough_volume))

# 3. Of those, how many look like real CTR opportunities under the starter rule
# (low_ctr_visible_page: impressions_90d >= 500, avg_position 1-20, ctr < 0.5)?
low_ctr_candidates = visible_enough_volume[visible_enough_volume["ctr"] < 0.5]
print("Low-CTR opportunity candidates:", len(low_ctr_candidates),
      f"({100 * len(low_ctr_candidates) / len(visible_enough_volume):.1f}% of eligible pages)")


Total rows: 30000
Rows with avg_position == 0 (no position data, must exclude): 1205

CTR by position tier (avg_position > 0 only):
               count      mean  median
position_tier                         
page_1         11814  0.652467    0.16
striking        7304  0.323239    0.11
page_3_5        7242  0.222484    0.03
deep            1319  0.150212    0.00
top_3           1116  2.764453    0.00

Visible pages with enough volume (position 1-20, impressions_90d >= 500): 12023
Low-CTR opportunity candidates: 9759 (81.2% of eligible pages)


## 4. Careful words: what I can and can't claim

**What I can claim (by the end of this lane):**
- *Observed*: which pages, in this 30,000-row starter slice (and later the warehouse), sit furthest below their position tier's expected CTR, with enough impression volume that the gap isn't noise.
- *Directional*: a ranked, reason-coded review queue that gives a reviewer a defensible order to work through, better than an unranked list or a flat CTR cutoff.
- *Decision-support*: "these pages are the best candidates to review for a title/meta/snippet rewrite first" — a prioritization tool, not a guarantee.

**What I will never claim:**
- That rewriting a flagged page's title or meta *will* increase its clicks — that requires an actual before/after experiment, not this dataset.
- Any Google ranking-algorithm factor, or that I've reverse-engineered how Google computes position or CTR.
- That a low-CTR page is broken — it could be seasonal, a SERP feature change, or genuinely low-volume noise, and I'll filter and flag for that rather than assume decline.
- Anything about a specific real client, URL, or query — all IDs here are pseudonyms, and I'll only ever report aggregated, pseudonymized results.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.